In [7]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import randint, loguniform
import statsmodels as stats
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import utils
skew_data = FunctionTransformer(utils.skew_data)
clean_data = FunctionTransformer(utils.preprocess)

In [3]:
path = "gs://data-epip/train/paysim.parquet"
train_pdf = pd.read_parquet(path, engine='pyarrow')


X_train, y_train = train_pdf.drop("isFraud", axis=1), train_pdf["isFraud"]
X_train.drop(["nameOrig", "nameDest", "step", "isFlaggedFraud"], axis=1, inplace=True)
num_col_name = X_train.select_dtypes(include=["float", "int"]).columns.tolist()
cat_col_name = X_train.select_dtypes(include=["category", "str", "object"]).columns.tolist()


In [22]:
num_pipeline = Pipeline(
    steps=[
        ("skew_data", skew_data),
        ("scaler", MinMaxScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder())
    ]
)

preprocess = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_col_name),
    ("cat", cat_pipeline, cat_col_name)
])

In [ ]:
# XgboostClassifier
# precision_recall vs roc_curve (TPR, FPR)

cv = StratifiedKFold(n_splits=3)

classifier = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr", # auprc
    tree_method="hist",
    random_state=42,
    n_jobs=1,
    early_stopping_rounds=10,
    use_label_encoder=True)

pipeline_xgb = ImbPipeline(
    steps=[
        ("clean_data", clean_data),
        ("preprocess", preprocess),
        ("smote", SMOTE(random_state=42)),
        ("classifier", classifier)
    ]
)

from scipy.stats import randint, loguniform, uniform

param_distributions = {
    "classifier__n_estimators": randint(200, 1200),
    "classifier__learning_rate": loguniform(0.01, 0.3),
    "classifier__max_depth": randint(3, 11),
    "classifier__min_child_weight": randint(1, 15),
    "classifier__subsample": uniform(0.6, 0.4),
    "classifier__colsample_bytree": uniform(0.6, 0.4),
    "classifier__reg_alpha": loguniform(1e-5, 10),
    "classifier__reg_lambda": loguniform(1e-3, 100),
    "classifier__gamma": loguniform(1e-5, 10),
    "classifier__max_bin": randint(64, 512),
}


random_search = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_distributions,
    n_iter=1,
    scoring="average_precision",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=8
)

In [ ]:
random_search.fit(X_train, y_train)

Fitting 10 folds for each of 30 candidates, totalling 300 fits


In [4]:
path = "gs://data-epip/test/paysim.parquet"
train_pdf = pd.read_parquet(path, engine='pyarrow')


X_test, y_test = train_pdf.drop("isFraud", axis=1), train_pdf["isFraud"]
X_test.drop(["nameOrig", "nameDest", "step", "isFlaggedFraud"], axis=1, inplace=True)


In [15]:
print(classification_report(y_train, random_search.predict(X_train)))

              precision    recall  f1-score   support

           0       1.00      0.61      0.76   5401246
           1       0.00      0.98      0.01      6981

    accuracy                           0.61   5408227
   macro avg       0.50      0.80      0.38   5408227
weighted avg       1.00      0.61      0.76   5408227



In [13]:
print(classification_report(y_test, random_search.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      0.84      0.91    953161
           1       0.01      0.88      0.01      1232

    accuracy                           0.84    954393
   macro avg       0.50      0.86      0.46    954393
weighted avg       1.00      0.84      0.91    954393



In [19]:
pd.Series(random_search.predict(X_test)).value_counts()

0    567107
1    387286
Name: count, dtype: int64

In [ ]:
joblib.dump(random_search.best_estimator_, '../output/xgboost-new.pkl')

['../output/xgboost.pkl']